# chess-llm-rl — Kaggle T4 training

Architecture:
- `/kaggle/input/.../chess-llm-rl/` — source code, **read-only, never copied**.
- `/kaggle/working/stockfish/` — Stockfish binary (downloaded fresh each session).
- `/kaggle/working/checkpoints/` — LoRA outputs (pushed to HF Hub).
- `/kaggle/working/` — working dir; `./stockfish/stockfish` and `./checkpoints/` resolve correctly from here.

**Kaggle Secrets required:** `WANDB_API_KEY`, `HF_TOKEN`, `HF_REPO`. Dataset `chess-llm-rl` must be attached.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install deps + make chess_rl importable via sys.path
# Unsloth recipe — --no-deps on trl/transformers/unsloth prevents transitive pulls of
# llm_blender and mergekit, which broke the previous (Gemma 3) setup on Kaggle.
import os, sys, importlib.util

DATASET_ROOT = '/kaggle/input/datasets/abhinavsachdeva7/chess-llm-rl/chess-llm-rl'
SRC = f'{DATASET_ROOT}/src'
assert os.path.exists(f'{SRC}/chess_rl/__init__.py'), f'src/chess_rl missing in {DATASET_ROOT}'

!pip install --upgrade -qqq uv
# Heavy base stack only if torch missing (Colab) — Kaggle already ships torch,
# so this branch usually no-ops and we fall through to the --no-deps upgrade line.
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

# This is the line that kills the llm_blender / mergekit dep-hell: --no-deps means
# pip installs only the names listed, without pulling transitive deps whose metadata
# was tripping trl's `is_*_available()` checks on Kaggle.
!uv pip install --upgrade --no-deps "transformers>=5.5.0" tokenizers "trl>=0.28.0" unsloth unsloth_zoo
!pip install --no-deps --upgrade timm  # Gemma 4 vision/audio — required for import

# Chess-project-specific deps not covered by Unsloth's stack.
!pip install -q python-chess wandb pydantic pyyaml datasets jinja2

if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + ':' + os.environ.get('PYTHONPATH', '')

import chess_rl, chess_rl.model, chess_rl.train
print('chess_rl loaded from:', chess_rl.__file__)

import trl
from trl import GRPOConfig, GRPOTrainer
print('trl version:', trl.__version__, '— GRPO import OK')

In [ ]:
# 3. Stockfish — download to /kaggle/working/stockfish/stockfish
import os, urllib.request, tarfile, glob, shutil
os.chdir('/kaggle/working')
SF_DIR = '/kaggle/working/stockfish'
SF_BIN = f'{SF_DIR}/stockfish'

# Always start clean — tar extracts a 'stockfish/' dir that collides with SF_BIN path.
if os.path.exists(SF_DIR):
    shutil.rmtree(SF_DIR)
os.makedirs(SF_DIR)

EXTRACT = f'{SF_DIR}/_extract'
os.makedirs(EXTRACT)
url = 'https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar'
tar_path = f'{SF_DIR}/sf.tar'
urllib.request.urlretrieve(url, tar_path)
with tarfile.open(tar_path) as t:
    t.extractall(EXTRACT, filter='data')
matches = glob.glob(f'{EXTRACT}/**/stockfish-ubuntu-x86-64-avx2', recursive=True)
assert matches, f'binary not found under {EXTRACT}'
shutil.move(matches[0], SF_BIN)
os.chmod(SF_BIN, 0o755)
shutil.rmtree(EXTRACT)
os.remove(tar_path)

print('stockfish at:', SF_BIN, 'exists:', os.path.exists(SF_BIN))
!{SF_BIN} bench 2>&1 | tail -3

In [ ]:
# 4. Prepare a runtime config in /kaggle/working (source config.yaml is read-only in input)
import os, yaml
os.chdir('/kaggle/working')
os.makedirs('checkpoints', exist_ok=True)

DATASET_ROOT = '/kaggle/input/datasets/abhinavsachdeva7/chess-llm-rl/chess-llm-rl'
with open(f'{DATASET_ROOT}/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override arm via env var if set (lets you launch two runs without editing the notebook)
arm_override = os.environ.get('ARM_OVERRIDE')  # 'fen_only' or 'fen_pgn'
if arm_override:
    cfg['training']['arm'] = arm_override

# Paths relative to cwd (/kaggle/working)
cfg['stockfish']['path'] = './stockfish/stockfish'
cfg['paths']['checkpoints'] = './checkpoints'

RUNTIME_CONFIG = '/kaggle/working/config_runtime.yaml'
with open(RUNTIME_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('runtime config:', RUNTIME_CONFIG)
print('model:', cfg['model']['name'])
print('max_completion_length:', cfg['grpo']['max_completion_length'])
print('arm:', cfg['training']['arm'])
print('cwd:', os.getcwd())

In [ ]:
# 5. W&B login
from kaggle_secrets import UserSecretsClient
import wandb
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
print('wandb login OK')

In [ ]:
# 6. HF login
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['HF_REPO'] = secrets.get_secret('HF_REPO')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('HF login OK, repo:', os.environ['HF_REPO'])

In [ ]:
# 7. Resume from HF Hub if a prior checkpoint exists
import os
from huggingface_hub import HfApi, snapshot_download

RESUME_DIR = None
try:
    api = HfApi()
    files = api.list_repo_files(os.environ['HF_REPO'])
    ckpts = sorted({f.split('/', 1)[0] for f in files if f.startswith('ckpt_games_')})
    if ckpts:
        latest = ckpts[-1]
        print('resuming from', latest)
        local = snapshot_download(repo_id=os.environ['HF_REPO'], allow_patterns=f'{latest}/*')
        RESUME_DIR = f'{local}/{latest}'
    else:
        print('no prior checkpoint — starting fresh')
except Exception as e:
    print('resume lookup failed (likely empty repo):', e)
print('RESUME_DIR =', RESUME_DIR)

## Phase B — model smoke (run before cell 8 the first time)

Verifies model loads, generates, and returns legal SAN. Skip on later sessions.

In [ ]:
# B1. Model smoke generate (S5 equivalent)
import torch, chess
from chess_rl.model import load_model
from chess_rl.prompts import build_messages, apply_template
from chess_rl.rewards import compute_reward, get_analyst, extract_san

model, tok = load_model()
print('device:', next(model.parameters()).device)
print('vram GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

b = chess.Board()
msgs = build_messages(b, [], arm='fen_only', llm_color=True)
prompt = apply_template(tok, msgs)
print('--- PROMPT TAIL ---'); print(prompt[-400:])

# Gemma 4 tok is a multimodal processor: first positional arg is `images`,
# so prompt must go through the `text=` kwarg or the processor sees text=None.
ids = tok(text=prompt, return_tensors='pt').to(model.device)
# thinking in hidden channel; visible response = <move>SAN</move> only (~10 tokens)
out = model.generate(**ids, max_new_tokens=32, do_sample=False, pad_token_id=tok.pad_token_id)
raw = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
resp = extract_san(raw)
print('--- RAW RESPONSE ---'); print(repr(raw))
print('--- EXTRACTED SAN ---'); print(repr(resp))
print('reward:', compute_reward(raw, b, get_analyst()))

In [ ]:
# B2. One full game vs Stockfish-400, no training (S8 equivalent)
import yaml, chess
from chess_rl.env import ChessEnvironment, play_full_game
from chess_rl.stockfish import StockfishManager

with open('/kaggle/working/config_runtime.yaml') as f:
    cfg = yaml.safe_load(f)

sf = StockfishManager(); sf.set_opponent_elo(400)
env = ChessEnvironment(cfg, sf, tokenizer=tok); env.reset(llm_color=chess.WHITE)

def gen(messages):
    prompt = apply_template(tok, messages)
    ids = tok(text=prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=32, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

result, metrics = play_full_game(env, gen)
print('result:', result)
print('metrics:', metrics)
total = max(1, metrics.legal + metrics.no_legal_fallback)
print('fallback rate:', round(metrics.no_legal_fallback / total, 3))
sf.close()

## Phase D — GRPO training

Start with a 10-game smoke to confirm no OOM, then remove the `TARGET_GAMES` override for the full 180-game session.

In [ ]:
# 8. Run training — smoke first (10 games), then full session (180 games)
import os, gc, torch
os.environ['TARGET_GAMES'] = os.environ.get('TARGET_GAMES', '10')
os.chdir('/kaggle/working')

# Free Phase B model/tokenizer if they exist — otherwise train_main loads a 2nd copy.
for _v in ('model', 'tok', 'env', 'sf', 'ids', 'out'):
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()
print('pre-train VRAM GB:', round(torch.cuda.memory_allocated()/1e9, 2))

from chess_rl.train import main as train_main
train_main('/kaggle/working/config_runtime.yaml', resume_from=RESUME_DIR)

In [ ]:
# 9. List produced checkpoints (final push happens inside save_checkpoint)
import os
for d in sorted(os.listdir('/kaggle/working/checkpoints')):
    if d.startswith('ckpt_games_'):
        print(d)